# Contrast Consistent Search (Burns et al. 2022)

**Paper**: Burns, Ye, Klein, Steinhardt 2022, *Discovering Latent Knowledge in Language Models Without Supervision* — [arxiv:2212.03827](https://arxiv.org/abs/2212.03827).

**Canonical repo**: [collin-burns/discovering_latent_knowledge](https://github.com/collin-burns/discovering_latent_knowledge) (frozen 2023).  
**Active fork**: [EleutherAI/elk](https://github.com/EleutherAI/elk) (maintained through 2024).

**Critique (read this)**: Farquhar, Varma, Lindner, Krueger et al. 2023, *Challenges with unsupervised LLM knowledge discovery* — [arxiv:2312.10029](https://arxiv.org/abs/2312.10029). Shows CCS is unstable and recovers *prominent features*, which are not necessarily **truth**. They prove that a near-identical objective is satisfied by many non-truth features and that CCS is no better than simple unsupervised baselines.

**Follow-up**: Mallen et al. 2023, *Eliciting Latent Knowledge from Quirky Language Models* — [arxiv:2312.01037](https://arxiv.org/abs/2312.01037).

---

> **Disclaimer.** CCS recovers *prominent* features; Farquhar 2023 showed this isn't always truth. We therefore ship a **difference-of-means** sanity baseline alongside CCS, and a **supervised logistic-regression** ceiling. If diff-of-means matches or beats CCS, CCS adds no unsupervised value for this task. We report all three honestly.

In [ ]:
# Install (quiet). Colab T4 target.
!pip -q install --upgrade transformers accelerate safetensors scikit-learn pandas matplotlib tqdm datasets

## Config

In [ ]:
import os, json, random, math
import numpy as np
import torch

MODEL_ID   = 'google/gemma-2-2b'
LAYER      = 14            # mid-late residual; sweep later
DATASET    = 'imdb'        # binary sentiment as clean CCS demo; set to 'truthful_qa' to use the harder (but noisier) TQA mc1 split.
N_TRAIN    = 400
N_TEST     = 100
CCS_LR     = 1e-3
CCS_STEPS  = 1000
CCS_RESTARTS = 10          # Burns et al. recommend multiple random inits; keep best by train loss.
SEED       = 42
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE      = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
MAX_LEN    = 192
SWEEP_LAYERS = [6, 10, 14, 18, 22]   # for per-layer AUROC sweep
OUT_DIR    = './ccs_out'
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print(f'Device={DEVICE} dtype={DTYPE} layer={LAYER} dataset={DATASET}')

## Build contrast pairs

For every labelled example we construct two statements:
* `text_yes = q + "\nAnswer: Yes"`
* `text_no  = q + "\nAnswer: No"`

The gold label tells us which one is true; CCS must learn this **without** seeing labels.

In [ ]:
from datasets import load_dataset

def build_pairs_imdb(n):
    """IMDB binary sentiment framed as 'Is this review positive?' Yes/No."""
    ds = load_dataset('imdb', split='train').shuffle(seed=SEED)
    pairs = []
    for ex in ds:
        review = ex['text'].strip().replace('<br />', ' ')
        if len(review) < 30:
            continue
        review = review[:600]
        q = f'Review: "{review}"\nQuestion: Is this review positive?'
        # label 1 = positive -> gold answer = Yes
        y = int(ex['label'])
        pairs.append({
            'text_yes': q + '\nAnswer: Yes',
            'text_no':  q + '\nAnswer: No',
            'y_true':   y,        # 1 iff "Yes" is the true answer
        })
        if len(pairs) >= n:
            break
    return pairs

def build_pairs_truthfulqa(n):
    """TruthfulQA mc1: pick the correct answer + a random wrong one.
    We form q+correct as the 'Yes-is-true' case and q+wrong as the 'No-is-true' case
    by randomising which side carries the true completion."""
    ds = load_dataset('truthful_qa', 'multiple_choice', split='validation').shuffle(seed=SEED)
    pairs = []
    for ex in ds:
        q = ex['question']
        choices = ex['mc1_targets']['choices']
        labels  = ex['mc1_targets']['labels']
        try:
            correct = choices[labels.index(1)]
        except ValueError:
            continue
        wrongs = [c for c, l in zip(choices, labels) if l == 0]
        if not wrongs:
            continue
        wrong = random.choice(wrongs)
        # Randomly assign which completion gets the 'Yes' slot.
        if random.random() < 0.5:
            text_yes = f'Q: {q}\nA: {correct}\nIs this answer correct?\nAnswer: Yes'
            text_no  = f'Q: {q}\nA: {correct}\nIs this answer correct?\nAnswer: No'
            y = 1
        else:
            text_yes = f'Q: {q}\nA: {wrong}\nIs this answer correct?\nAnswer: Yes'
            text_no  = f'Q: {q}\nA: {wrong}\nIs this answer correct?\nAnswer: No'
            y = 0
        pairs.append({'text_yes': text_yes, 'text_no': text_no, 'y_true': y})
        if len(pairs) >= n:
            break
    return pairs

builder = build_pairs_imdb if DATASET == 'imdb' else build_pairs_truthfulqa
all_pairs = builder(N_TRAIN + N_TEST)
assert len(all_pairs) >= N_TRAIN + N_TEST, f'Need {N_TRAIN+N_TEST}, got {len(all_pairs)}'
train_pairs = all_pairs[:N_TRAIN]
test_pairs  = all_pairs[N_TRAIN:N_TRAIN+N_TEST]

print(f'train={len(train_pairs)} test={len(test_pairs)}')
print(f'train pos-rate={np.mean([p["y_true"] for p in train_pairs]):.3f}')
print(f'test  pos-rate={np.mean([p["y_true"] for p in test_pairs]):.3f}')
print('Example train[0].text_yes[:200]:\n', train_pairs[0]['text_yes'][:200])

## Capture hidden states

Forward every contrast sentence, pull the layer-`LAYER` residual hidden state at the last token, subtract the per-class mean (Burns' normalisation — removes the trivial "Yes vs No token" direction so CCS/diff-of-means can't cheat).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map=DEVICE,
    attn_implementation='sdpa',   # no flash-attn
    output_hidden_states=True,
)
model.eval()
D_MODEL = model.config.hidden_size
N_LAYERS = model.config.num_hidden_layers
print(f'Loaded {MODEL_ID} | d_model={D_MODEL} | layers={N_LAYERS}')
assert 0 <= LAYER <= N_LAYERS, f'LAYER {LAYER} out of range for {N_LAYERS} layers'

In [ ]:
@torch.no_grad()
def batched_hidden(texts, layers, batch_size=8):
    """Return dict{layer_idx: np.ndarray (N, d_model)} last-token hidden."""
    out = {L: [] for L in layers}
    for i in tqdm(range(0, len(texts), batch_size), desc='fwd'):
        chunk = texts[i:i+batch_size]
        enc = tok(chunk, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(DEVICE)
        outputs = model(**enc, output_hidden_states=True, use_cache=False)
        # hidden_states: tuple of (num_layers+1) tensors (B, T, D) — index 0 is embeddings.
        # last-token = last non-pad position.
        lengths = enc['attention_mask'].sum(dim=1) - 1  # (B,)
        for L in layers:
            h = outputs.hidden_states[L]  # (B, T, D)
            last = h[torch.arange(h.size(0)), lengths]  # (B, D)
            out[L].append(last.float().cpu().numpy())
    return {L: np.concatenate(v, axis=0) for L, v in out.items()}

texts_yes_train = [p['text_yes'] for p in train_pairs]
texts_no_train  = [p['text_no']  for p in train_pairs]
texts_yes_test  = [p['text_yes'] for p in test_pairs]
texts_no_test   = [p['text_no']  for p in test_pairs]
y_train = np.array([p['y_true'] for p in train_pairs], dtype=np.int64)
y_test  = np.array([p['y_true'] for p in test_pairs],  dtype=np.int64)

layers_to_probe = sorted(set(SWEEP_LAYERS + [LAYER]))
H_yes_train = batched_hidden(texts_yes_train, layers_to_probe)
H_no_train  = batched_hidden(texts_no_train,  layers_to_probe)
H_yes_test  = batched_hidden(texts_yes_test,  layers_to_probe)
H_no_test   = batched_hidden(texts_no_test,   layers_to_probe)

def normalise(h_yes_tr, h_no_tr, h_yes_te, h_no_te):
    """Burns normalisation: subtract per-class mean computed on TRAIN."""
    mu_yes = h_yes_tr.mean(0, keepdims=True)
    mu_no  = h_no_tr.mean(0, keepdims=True)
    return (h_yes_tr - mu_yes, h_no_tr - mu_no, h_yes_te - mu_yes, h_no_te - mu_no)

print('hidden shapes @ chosen layer:', H_yes_train[LAYER].shape, H_no_train[LAYER].shape)

## CCS loss and training

CCS probe: `p = sigmoid(W h + b)` (scalar). Loss on paired hiddens `(h+, h-)`:

$$\mathcal{L} = \underbrace{(p^+ + p^- - 1)^2}_{\text{consistency}} + \underbrace{\min(p^+, p^-)^2}_{\text{confidence}}$$

Best-of-`CCS_RESTARTS` random inits (Burns et al.), pick by train loss. At test: predict `argmax(p_yes, 1 - p_no)` → Yes if the probe thinks the Yes-completion is true and the No-completion is false. We report both `acc` vs gold and `acc` after an unsupervised sign-flip (since CCS is direction-ambiguous).

In [ ]:
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score

class CCSProbe(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    def forward(self, h):
        return torch.sigmoid(self.linear(h)).squeeze(-1)

def ccs_loss(p_yes, p_no):
    consistency = (p_yes + p_no - 1).pow(2).mean()
    confidence  = torch.stack([p_yes, p_no], dim=-1).min(dim=-1).values.pow(2).mean()
    return consistency + confidence

def train_ccs(Xy, Xn, d, steps=CCS_STEPS, lr=CCS_LR, restarts=CCS_RESTARTS, device=DEVICE):
    Xy_t = torch.from_numpy(Xy).float().to(device)
    Xn_t = torch.from_numpy(Xn).float().to(device)
    best_probe, best_loss = None, float('inf')
    for r in range(restarts):
        torch.manual_seed(SEED + r)
        probe = CCSProbe(d).to(device)
        opt = torch.optim.Adam(probe.parameters(), lr=lr)
        for _ in range(steps):
            opt.zero_grad()
            p_y = probe(Xy_t); p_n = probe(Xn_t)
            loss = ccs_loss(p_y, p_n)
            loss.backward(); opt.step()
        final = loss.item()
        if final < best_loss:
            best_loss, best_probe = final, probe
    return best_probe, best_loss

@torch.no_grad()
def ccs_scores(probe, Xy, Xn):
    Xy_t = torch.from_numpy(Xy).float().to(DEVICE)
    Xn_t = torch.from_numpy(Xn).float().to(DEVICE)
    p_y = probe(Xy_t).cpu().numpy()
    p_n = probe(Xn_t).cpu().numpy()
    # Burns: score for 'Yes is correct' = 0.5 * (p_y + (1 - p_n))
    return 0.5 * (p_y + (1.0 - p_n)), p_y, p_n

def eval_ccs(probe, Xy_tr, Xn_tr, Xy_te, Xn_te, y_tr, y_te):
    s_tr, _, _ = ccs_scores(probe, Xy_tr, Xn_tr)
    s_te, _, _ = ccs_scores(probe, Xy_te, Xn_te)
    # Sign-flip on TRAIN only (unsupervised heuristic: higher mean score should correspond to y=1).
    if roc_auc_score(y_tr, s_tr) < 0.5:
        s_tr = 1.0 - s_tr; s_te = 1.0 - s_te
    auroc = roc_auc_score(y_te, s_te)
    acc   = accuracy_score(y_te, (s_te > 0.5).astype(int))
    return acc, auroc

# Train and evaluate CCS at the main LAYER.
Xy_tr, Xn_tr, Xy_te, Xn_te = normalise(H_yes_train[LAYER], H_no_train[LAYER], H_yes_test[LAYER], H_no_test[LAYER])
probe, ccs_train_loss = train_ccs(Xy_tr, Xn_tr, D_MODEL)
ccs_acc, ccs_auroc = eval_ccs(probe, Xy_tr, Xn_tr, Xy_te, Xn_te, y_train, y_test)
print(f'CCS @ layer {LAYER}: acc={ccs_acc:.3f}  auroc={ccs_auroc:.3f}  train_loss={ccs_train_loss:.4f}')

## Difference-of-means baseline (Farquhar-required)

The simplest unsupervised direction: `v = mean(h_yes) - mean(h_no)` on TRAIN. Project and threshold at 0. If this matches/beats CCS, CCS is not extracting anything beyond the `yes/no` prompt contrast — exactly Farquhar's 2023 point.

In [ ]:
def diff_of_means(Xy_tr, Xn_tr, Xy_te, Xn_te, y_tr, y_te):
    v = Xy_tr.mean(0) - Xn_tr.mean(0)
    v = v / (np.linalg.norm(v) + 1e-8)
    # score = how much the YES-side projects onto v minus the NO-side projection
    score_tr = (Xy_tr @ v) - (Xn_tr @ v)
    score_te = (Xy_te @ v) - (Xn_te @ v)
    if roc_auc_score(y_tr, score_tr) < 0.5:
        score_tr = -score_tr; score_te = -score_te
    auroc = roc_auc_score(y_te, score_te)
    acc   = accuracy_score(y_te, (score_te > 0).astype(int))
    return acc, auroc

dom_acc, dom_auroc = diff_of_means(Xy_tr, Xn_tr, Xy_te, Xn_te, y_train, y_test)
print(f'diff-of-means @ layer {LAYER}: acc={dom_acc:.3f}  auroc={dom_auroc:.3f}')

## Supervised logistic-regression ceiling

Fit LR on `(h_yes - h_no, y_true)` with gold labels. This is the supervised upper bound; CCS should be compared against both this ceiling and the diff-of-means floor.

In [ ]:
from sklearn.linear_model import LogisticRegression

def supervised_lr(Xy_tr, Xn_tr, Xy_te, Xn_te, y_tr, y_te):
    X_tr = Xy_tr - Xn_tr
    X_te = Xy_te - Xn_te
    clf = LogisticRegression(max_iter=1000, C=1.0).fit(X_tr, y_tr)
    score_te = clf.decision_function(X_te)
    auroc = roc_auc_score(y_te, score_te)
    acc   = accuracy_score(y_te, clf.predict(X_te))
    return acc, auroc

lr_acc, lr_auroc = supervised_lr(Xy_tr, Xn_tr, Xy_te, Xn_te, y_train, y_test)
print(f'LR supervised @ layer {LAYER}: acc={lr_acc:.3f}  auroc={lr_auroc:.3f}')

## Results — per-layer sweep + comparison table + honest commentary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sweep_rows = []
for L in SWEEP_LAYERS:
    Xy_tr_L, Xn_tr_L, Xy_te_L, Xn_te_L = normalise(H_yes_train[L], H_no_train[L], H_yes_test[L], H_no_test[L])
    probe_L, _ = train_ccs(Xy_tr_L, Xn_tr_L, D_MODEL)
    ccs_a, ccs_r = eval_ccs(probe_L, Xy_tr_L, Xn_tr_L, Xy_te_L, Xn_te_L, y_train, y_test)
    dom_a, dom_r = diff_of_means(Xy_tr_L, Xn_tr_L, Xy_te_L, Xn_te_L, y_train, y_test)
    lr_a,  lr_r  = supervised_lr(Xy_tr_L, Xn_tr_L, Xy_te_L, Xn_te_L, y_train, y_test)
    sweep_rows.append({'layer': L,
                       'ccs_acc': ccs_a, 'ccs_auroc': ccs_r,
                       'dom_acc': dom_a, 'dom_auroc': dom_r,
                       'lr_acc':  lr_a,  'lr_auroc':  lr_r})
sweep = pd.DataFrame(sweep_rows).set_index('layer')
print(sweep.round(3))

# Headline table at chosen LAYER.
headline = pd.DataFrame([
    {'method': 'CCS',            'accuracy': ccs_acc, 'AUROC': ccs_auroc, 'notes': 'unsupervised (Burns 2022)'},
    {'method': 'diff-of-means',  'accuracy': dom_acc, 'AUROC': dom_auroc, 'notes': 'unsupervised baseline (Farquhar 2023)'},
    {'method': 'LR supervised',  'accuracy': lr_acc,  'AUROC': lr_auroc,  'notes': 'supervised ceiling'},
])
print(f'\n=== Headline @ layer {LAYER} ===')
print(headline.to_string(index=False))

# Persist results.
results = {
    'model_id': MODEL_ID, 'dataset': DATASET, 'layer': LAYER,
    'n_train': N_TRAIN, 'n_test': N_TEST,
    'headline': headline.to_dict(orient='records'),
    'sweep': sweep.reset_index().to_dict(orient='records'),
    'ccs_train_loss': ccs_train_loss,
    'config': {'ccs_lr': CCS_LR, 'ccs_steps': CCS_STEPS, 'ccs_restarts': CCS_RESTARTS, 'seed': SEED},
}
with open(os.path.join(OUT_DIR, 'ccs_comparison.json'), 'w') as f:
    json.dump(results, f, indent=2)

# Plot sweep.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep.index, sweep['ccs_auroc'],  marker='o', label='CCS')
ax.plot(sweep.index, sweep['dom_auroc'],  marker='s', label='diff-of-means')
ax.plot(sweep.index, sweep['lr_auroc'],   marker='^', label='LR supervised')
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='chance')
ax.set_xlabel('layer index'); ax.set_ylabel('test AUROC')
ax.set_title(f'CCS vs diff-of-means vs LR | {MODEL_ID} | {DATASET}')
ax.set_ylim(0.4, 1.01); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'ccs_chart.png'), dpi=140)
plt.show()
print(f'Saved: {os.path.join(OUT_DIR, "ccs_comparison.json")}')
print(f'Saved: {os.path.join(OUT_DIR, "ccs_chart.png")}')

# Honest commentary.
gap = ccs_auroc - dom_auroc
print('\n=== Honest commentary ===')
if gap <= 0.01:
    verdict = (
        f'CCS AUROC ({ccs_auroc:.3f}) does NOT exceed diff-of-means ({dom_auroc:.3f}) '
        f'(gap={gap:+.3f}). For this task, CCS adds no unsupervised value over the '
        f'trivial Yes/No prompt-contrast direction. This is exactly the failure mode '
        f'Farquhar et al. 2023 (arxiv:2312.10029) flagged: CCS recovers the most '
        f'prominent feature, not truth.'
    )
elif gap < 0.05:
    verdict = (
        f'CCS ({ccs_auroc:.3f}) modestly beats diff-of-means ({dom_auroc:.3f}) by {gap:+.3f}. '
        f'The gap is within the instability range Farquhar et al. 2023 reported across '
        f'random inits; treat this as weak evidence and re-run with different seeds/datasets '
        f'before claiming CCS recovered truth.'
    )
else:
    verdict = (
        f'CCS ({ccs_auroc:.3f}) beats diff-of-means ({dom_auroc:.3f}) by {gap:+.3f}, and the '
        f'supervised ceiling is {lr_auroc:.3f}. CCS is extracting structure beyond the prompt '
        f'contrast on THIS dataset. Farquhar-style caveats still apply: a prominent-but-not-truth '
        f'feature may be correlated with the label here; validate on quirky-LM style adversarial '
        f'probes (Mallen et al. 2023) before claiming latent-knowledge extraction.'
    )
print(verdict)

## Farquhar 2023 critique — read before trusting these numbers

Farquhar, Varma, Lindner, Krueger et al. 2023, [*Challenges with unsupervised LLM knowledge discovery*](https://arxiv.org/abs/2312.10029), empirically show:

1. **CCS is unstable across random inits.** A non-trivial fraction of training runs converge to different directions with different accuracies. We mitigate with `CCS_RESTARTS` best-of-N but do not eliminate it.
2. **CCS recovers *prominent* features, not necessarily *truth*.** Any feature that is roughly logically consistent across the `(yes, no)` contrast pair and confident will satisfy the objective. On synthetic "quirky" datasets where the model knows the truth but the prompt pushes a lie, CCS often follows the prompt rather than the knowledge.
3. **Diff-of-means is a strong, frequently equivalent baseline.** Our diff-of-means column exists precisely because Farquhar showed the trivial unsupervised direction often matches CCS. If our gap is ≤ ~0.01 AUROC, we should NOT interpret CCS as having found a "truth" direction.
4. **Supervised probes are an upper bound, not a solution.** LR with gold labels is included as a ceiling only; it requires supervision that real ELK would not have.

**Action items for any downstream CCS claim:**
- Report the diff-of-means baseline alongside every CCS number.
- Report multi-seed mean ± std (we use best-of-10 inits; for paper-grade, add a std over 5 outer seeds).
- Test on *quirky* / persona-conflict datasets (Mallen et al. 2023, [arxiv:2312.01037](https://arxiv.org/abs/2312.01037)) before calling a probe "truth".